# QUEDA — Fiscalização de dados abertos do Crea-SP

Executa a coleta e a análise **a partir do Colab** (o tráfego sai por IPs do Google, não da sua máquina).

**Os dados ficam no Google Drive** (célula 1). O Colab desconecta e apaga o disco; o Drive faz a coleta sobreviver e a célula 4 retoma de onde parou.

**Camada de IA (opcional)**: roda um modelo local via Ollama dentro do próprio Colab — sem chave de API e sem que nenhum dado do conselho saia da máquina. Pular a célula 3 não tira nenhum número do relatório.

> Aviso: todo achado é **indício**, não prova. Confirme na fonte antes de qualquer denúncia.


In [ ]:
# 1) Google Drive — é isto que faz a coleta sobreviver à queda de sessão
# O Colab desconecta e apaga o disco. O clone do repositório é descartável
# (refazer leva segundos), mas `data/raw` guarda HORAS de coleta com throttle
# e `data/cache` guarda o que o modelo local já leu. Os dois vão para o Drive.
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA = '/content/drive/MyDrive/queda'          # mude se quiser outro lugar
os.environ['QUEDA_DATA_DIR']   = f'{PASTA}/data'
os.environ['QUEDA_OUTPUT_DIR'] = f'{PASTA}/output'
os.environ['OLLAMA_MODELS']    = f'{PASTA}/ollama'   # evita rebaixar 4,7 GB
for p in ('data/raw', 'data/staged', 'data/cache', 'output', 'ollama'):
    os.makedirs(f'{PASTA}/{p}', exist_ok=True)
print('persistido em', PASTA)
!du -sh {PASTA}/data 2>/dev/null || true


In [ ]:
# 2) Obter o código (efêmero: o Drive guarda os dados, não o código)
# O repositório é PRIVADO, então o clone pede um token. Gere um fine-grained
# token com `Contents: Read-only` em github.com/settings/personal-access-tokens
# e cole no prompt — ele NÃO fica salvo no notebook nem no Drive.
import os, subprocess
REPO = 'github.com/caiofvserra/queda.git'
BRANCH = 'claude/crea-sp-audit-tool-m9nw7t'
if os.path.exists('/content/queda'):
    !cd /content/queda && git fetch -q origin {BRANCH} && git reset -q --hard origin/{BRANCH}
else:
    r = subprocess.run(['git','clone','--depth','1','-b',BRANCH,f'https://{REPO}'],
                       cwd='/content')
    if r.returncode != 0:
        from getpass import getpass
        tok = getpass('Token GitHub (Contents: read-only): ')
        subprocess.run(['git','clone','--depth','1','-b',BRANCH,
                        f'https://{tok}@{REPO}'], cwd='/content', check=True)
        del tok
%cd /content/queda
!pip -q install -r requirements.txt
!pip -q install networkx   # intermediação exata no grafo da rede


In [ ]:
# 3) OPCIONAL — camada de IA rodando LOCAL (Ollama)
# Nenhum número do relatório depende disto: todo cálculo é determinístico.
# O modelo entra só onde regra nenhuma serve — enquadrar o resíduo da
# taxonomia, ler a prosa das atas e redigir o resumo executivo.
#
# Roda 100% na máquina do Colab: nenhum dado do conselho sai daqui, e não há
# chave nem custo por token. Com OLLAMA_MODELS no Drive (célula 1), o download
# de ~4,7 GB acontece UMA vez e as próximas sessões só carregam.
#
# Ative a GPU antes: Ambiente de execução -> Alterar tipo -> GPU (T4).
MODELO = 'qwen2.5:7b-instruct'   # leve: 'llama3.2:3b' | melhor: 'gemma2:9b'

import os, subprocess, time
os.environ['QUEDA_LLM_MODELO'] = MODELO

!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 env={**os.environ})
time.sleep(5)
!python main.py llm


In [ ]:
# 4) Coleta + análise + relatório
# --desde auto: sonda anos retroativamente até a API esvaziar (meta: pré-2016)
# --retomar:    pula as etapas que já vieram — deixe SEMPRE ligado, é o que
#               torna barata a volta depois de uma desconexão.
# Se cair no meio, rode ESTA MESMA célula de novo: ela continua de onde parou.
!python main.py all --desde auto --ate 2026 --retomar


In [ ]:
# 5) Ver o relatório aqui mesmo
import os
from IPython.display import HTML, display
display(HTML(open(os.path.join(os.environ['QUEDA_OUTPUT_DIR'], 'report.html')).read()))


In [ ]:
# 6) O relatório e os CSVs já estão no Drive (pasta `output`).
# Esta célula só baixa uma cópia zipada para a sua máquina.
import os
SAIDA = os.environ['QUEDA_OUTPUT_DIR']
!cd "$SAIDA/.." && zip -qr /content/resultado_queda.zip output/
from google.colab import files
files.download('/content/resultado_queda.zip')


## Quando a sessão cair (e ela vai cair)

Reconecte e rode as células **1 → 2 → 4** de novo (a 3 só se quiser a IA). A célula 4 pula tudo que já está no Drive e continua de onde parou — não recomeça a coleta.

- `data/raw/` guarda a resposta de cada requisição; nada é rebaixado.
- Etapa que veio **vazia** é tentada de novo de propósito: dataset zerado quase sempre é bloqueio por rajada (403), e 403 nunca é cacheado.
- Para forçar uma recoleta limpa de um período, apague os arquivos correspondentes em `data/raw/` (ou rode sem `--retomar`).

Só análise, sem tocar na rede: `!python main.py analyze` (aceita `--sem-llm`).

A aba **Diagnóstico da coleta** do relatório diz o que veio, o que faltou e qual análise ficou desligada por falta de qual fonte — comece por ela.
